# Estudo: Previsão da Taxa de Corrosão em Dutos

Este notebook demonstra como utilizar modelos de Machine Learning para prever a **taxa de crescimento de corrosão** em anomalias detectadas por passagem de PIG (Pipeline Inspection Gauge).

**Abordagem:**
1. Carregar e limpar os dados reais da inspeção
2. Simular inspeções futuras com modelo de crescimento realista
3. Treinar e comparar modelos: Regressão Linear, Random Forest e XGBoost
4. Avaliar qual modelo melhor prevê a taxa de corrosão

## 1. Importação de Bibliotecas e Carregamento dos Dados

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
raw = pd.read_excel('exemplo passagem de pig.xlsx')

colunas = raw.iloc[0].values
df = raw.iloc[1:].copy()
df.columns = [
    'id_tubo', 'posicao_m', 'dist_sold_ant_m', 'compr_tubo_m',
    'i_e', 'tipo', 'posicao_horaria', 'espessura_mm',
    'comprimento_mm', 'largura_mm', 'profundidade_pct', 'erf', 'tipo_pof'
]

print(f'Colunas originais do Excel: {list(colunas)}')
print(f'\nDataset: {df.shape[0]} anomalias, {df.shape[1]} colunas')
df.head()

## 2. Limpeza e Preparação dos Dados

In [ ]:
def limpar_numero(valor):
    """Converte string com formato brasileiro (1.234,56) para float."""
    if isinstance(valor, str):
        return float(valor.replace('.', '').replace(',', '.'))
    return float(valor)

colunas_numericas = [
    'posicao_m', 'dist_sold_ant_m', 'compr_tubo_m',
    'espessura_mm', 'comprimento_mm', 'largura_mm',
    'profundidade_pct', 'erf'
]

for col in colunas_numericas:
    df[col] = df[col].apply(limpar_numero)

df['i_e'] = df['i_e'].replace('-', 'E')

df.reset_index(drop=True, inplace=True)
print('Tipos de dados após limpeza:')
print(df.dtypes)
print(f'\nValores nulos: {df.isnull().sum().sum()}')

## 3. Análise Exploratória dos Dados (EDA)

In [ ]:
df[colunas_numericas].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Distribuição das Variáveis das Anomalias', fontsize=14, fontweight='bold')

vars_plot = ['profundidade_pct', 'comprimento_mm', 'largura_mm', 'espessura_mm', 'erf', 'posicao_m']
titles = ['Profundidade (%)', 'Comprimento (mm)', 'Largura (mm)', 'Espessura (mm)', 'ERF', 'Posição (m)']

for ax, var, title in zip(axes.flat, vars_plot, titles):
    sns.histplot(df[var], kde=True, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(axes, ['tipo', 'tipo_pof', 'i_e'],
                           ['Tipo de Anomalia', 'Tipo POF', 'Interna/Externa']):
    df[col].value_counts().plot.bar(ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
cols_corr = ['profundidade_pct', 'comprimento_mm', 'largura_mm', 'espessura_mm', 'erf']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[cols_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True)
ax.set_title('Correlação entre Variáveis Numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Simulação de Inspeções Futuras

Como temos apenas **uma inspeção**, vamos simular o crescimento das anomalias ao longo do tempo para criar o dataset de treino. O modelo de crescimento segue premissas realistas:

- A taxa de corrosão depende do tipo de anomalia, profundidade atual e espessura
- Anomalias mais profundas tendem a crescer mais rápido
- Corrosão externa (CORR) geralmente cresce diferente de defeitos de fabricação
- Adicionamos ruído aleatório para simular variabilidade real

> **Nota:** Quando os dados reais de múltiplas inspeções estiverem disponíveis, basta substituir esta etapa pelo pareamento real das anomalias entre inspeções.

In [ ]:
np.random.seed(42)

anos_inspecao = [0, 5, 10, 15]

taxa_base = {
    'CORR': 0.8,
    'ASCI': 0.4,
    'COSC': 0.3,
}

registros = []

for idx, row in df.iterrows():
    prof_atual = row['profundidade_pct']
    tipo = row['tipo']
    esp = row['espessura_mm']
    compr = row['comprimento_mm']
    larg = row['largura_mm']

    base = taxa_base.get(tipo, 0.5)
    fator_prof = 1 + (prof_atual / 100)
    fator_esp = 8.0 / esp

    for i in range(len(anos_inspecao) - 1):
        ano_ini = anos_inspecao[i]
        ano_fim = anos_inspecao[i + 1]
        intervalo = ano_fim - ano_ini

        ruido = np.random.normal(1.0, 0.15)
        taxa_real = base * fator_prof * fator_esp * ruido
        taxa_real = max(taxa_real, 0.05)

        crescimento = taxa_real * intervalo
        prof_futura = min(prof_atual + crescimento, 95)

        registros.append({
            'id_tubo': row['id_tubo'],
            'tipo': tipo,
            'tipo_pof': row['tipo_pof'],
            'i_e': row['i_e'],
            'espessura_mm': esp,
            'comprimento_mm': compr,
            'largura_mm': larg,
            'profundidade_inicial_pct': prof_atual,
            'profundidade_final_pct': round(prof_futura, 2),
            'erf': row['erf'],
            'ano_inicio': ano_ini,
            'ano_fim': ano_fim,
            'intervalo_anos': intervalo,
            'taxa_corrosao_pct_ano': round(taxa_real, 4)
        })

        prof_atual = prof_futura
        fator_prof = 1 + (prof_atual / 100)

df_ml = pd.DataFrame(registros)
print(f'Dataset para ML: {df_ml.shape[0]} registros')
print(f'Taxa de corrosão - média: {df_ml["taxa_corrosao_pct_ano"].mean():.2f} %/ano')
print(f'Taxa de corrosão - min: {df_ml["taxa_corrosao_pct_ano"].min():.2f}, max: {df_ml["taxa_corrosao_pct_ano"].max():.2f}')
df_ml.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df_ml['taxa_corrosao_pct_ano'], kde=True, ax=axes[0],
             color='coral', edgecolor='white', bins=40)
axes[0].set_title('Distribuição da Taxa de Corrosão (%/ano)', fontweight='bold')
axes[0].set_xlabel('Taxa de Corrosão (%/ano)')

sns.boxplot(data=df_ml, x='tipo', y='taxa_corrosao_pct_ano', ax=axes[1],
            palette='Set2')
axes[1].set_title('Taxa de Corrosão por Tipo de Anomalia', fontweight='bold')
axes[1].set_xlabel('Tipo')
axes[1].set_ylabel('Taxa (%/ano)')

plt.tight_layout()
plt.show()

## 5. Preparação das Features para os Modelos

In [ ]:
le_tipo = LabelEncoder()
le_pof = LabelEncoder()
le_ie = LabelEncoder()

df_ml['tipo_cod'] = le_tipo.fit_transform(df_ml['tipo'])
df_ml['tipo_pof_cod'] = le_pof.fit_transform(df_ml['tipo_pof'])
df_ml['i_e_cod'] = le_ie.fit_transform(df_ml['i_e'])

features = [
    'espessura_mm', 'comprimento_mm', 'largura_mm',
    'profundidade_inicial_pct', 'erf',
    'tipo_cod', 'tipo_pof_cod', 'i_e_cod', 'ano_inicio'
]

target = 'taxa_corrosao_pct_ano'

X = df_ml[features]
y = df_ml[target]

print(f'Features ({len(features)}): {features}')
print(f'Target: {target}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

## 6. Treinamento e Comparação dos Modelos

Vamos treinar 3 modelos e compará-los usando **validação cruzada (5-fold)**:

In [ ]:
modelos = {
    'Regressão Linear': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
resultados = {}

for nome, modelo in modelos.items():
    scores_r2 = cross_val_score(modelo, X, y, cv=kf, scoring='r2')
    scores_mae = -cross_val_score(modelo, X, y, cv=kf, scoring='neg_mean_absolute_error')
    scores_rmse = np.sqrt(-cross_val_score(modelo, X, y, cv=kf, scoring='neg_mean_squared_error'))

    resultados[nome] = {
        'R²': scores_r2.mean(),
        'R² (std)': scores_r2.std(),
        'MAE': scores_mae.mean(),
        'RMSE': scores_rmse.mean()
    }

    print(f'\n{nome}:')
    print(f'  R²   = {scores_r2.mean():.4f} (±{scores_r2.std():.4f})')
    print(f'  MAE  = {scores_mae.mean():.4f}')
    print(f'  RMSE = {scores_rmse.mean():.4f}')

df_resultados = pd.DataFrame(resultados).T
print('\n')
df_resultados.style.highlight_max(subset=['R²'], color='lightgreen') \
                    .highlight_min(subset=['MAE', 'RMSE'], color='lightgreen')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metricas = ['R²', 'MAE', 'RMSE']
cores = ['#2ecc71', '#e74c3c', '#e74c3c']

for ax, metrica, cor in zip(axes, metricas, cores):
    valores = [resultados[m][metrica] for m in resultados]
    bars = ax.bar(resultados.keys(), valores, color=cor, edgecolor='white', alpha=0.85)
    ax.set_title(metrica, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Comparação dos Modelos (Validação Cruzada 5-Fold)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Importância das Features (Random Forest & XGBoost)

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X, y)

xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
xgb.fit(X, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, modelo_fit, nome in zip(axes, [rf, xgb], ['Random Forest', 'XGBoost']):
    importancias = pd.Series(modelo_fit.feature_importances_, index=features)
    importancias = importancias.sort_values(ascending=True)
    importancias.plot.barh(ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Importância das Features - {nome}', fontweight='bold')
    ax.set_xlabel('Importância')

plt.tight_layout()
plt.show()

## 8. Previsão vs Valor Real (Melhor Modelo)

In [ ]:
melhor_nome = max(resultados, key=lambda k: resultados[k]['R²'])
print(f'Melhor modelo: {melhor_nome}')

melhor_modelo = modelos[melhor_nome]
melhor_modelo.fit(X, y)
y_pred = melhor_modelo.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y, y_pred, alpha=0.3, s=15, color='steelblue')
lim = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Ideal (y = ŷ)')
axes[0].set_xlabel('Taxa Real (%/ano)')
axes[0].set_ylabel('Taxa Prevista (%/ano)')
axes[0].set_title(f'{melhor_nome}: Previsto vs Real', fontweight='bold')
axes[0].legend()

residuos = y - y_pred
sns.histplot(residuos, kde=True, ax=axes[1], color='coral', edgecolor='white', bins=40)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Distribuição dos Resíduos', fontweight='bold')
axes[1].set_xlabel('Resíduo (Real - Previsto)')

plt.tight_layout()
plt.show()

## 9. Simulação: Evolução de uma Anomalia ao Longo do Tempo

In [ ]:
exemplo = df_ml[df_ml['tipo'] == 'CORR'].iloc[0]

anos_futuros = list(range(0, 31, 5))
prof_evolucao = [exemplo['profundidade_inicial_pct']]
prof_atual = exemplo['profundidade_inicial_pct']

print(f"Anomalia exemplo (tipo: {exemplo['tipo']}, tubo: {exemplo['id_tubo']})")
print(f"Profundidade inicial: {prof_atual:.1f}%\n")

for i in range(len(anos_futuros) - 1):
    entrada = pd.DataFrame([{
        'espessura_mm': exemplo['espessura_mm'],
        'comprimento_mm': exemplo['comprimento_mm'],
        'largura_mm': exemplo['largura_mm'],
        'profundidade_inicial_pct': prof_atual,
        'erf': exemplo['erf'],
        'tipo_cod': le_tipo.transform([exemplo['tipo']])[0],
        'tipo_pof_cod': le_pof.transform([exemplo['tipo_pof']])[0],
        'i_e_cod': le_ie.transform([exemplo['i_e']])[0],
        'ano_inicio': anos_futuros[i]
    }])

    taxa_prevista = melhor_modelo.predict(entrada)[0]
    intervalo = anos_futuros[i+1] - anos_futuros[i]
    prof_atual = min(prof_atual + taxa_prevista * intervalo, 100)
    prof_evolucao.append(round(prof_atual, 2))
    print(f"  Ano {anos_futuros[i+1]:>2}: taxa prevista = {taxa_prevista:.2f} %/ano → profundidade = {prof_atual:.1f}%")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(anos_futuros, prof_evolucao, 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax.axhline(y=80, color='orange', linestyle='--', linewidth=1.5, label='Limite crítico (80%)')
ax.fill_between(anos_futuros, prof_evolucao, alpha=0.15, color='red')
ax.set_xlabel('Anos', fontsize=12)
ax.set_ylabel('Profundidade (%)', fontsize=12)
ax.set_title('Evolução Prevista da Profundidade de uma Anomalia (CORR)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 100)
ax.set_xticks(anos_futuros)
plt.tight_layout()
plt.show()

## 10. Conclusão

### Resultados
- O modelo baseado em **ensemble de árvores** (Random Forest / XGBoost) apresentou desempenho superior à Regressão Linear para prever a taxa de corrosão.
- As features mais importantes para a previsão foram a **profundidade inicial**, **espessura** e **tipo de anomalia**.
- O modelo consegue projetar a evolução temporal de cada anomalia, permitindo estimar quando ela atingirá limites críticos.

### Próximos Passos
1. **Substituir dados simulados** por dados reais de múltiplas inspeções
2. **Parear anomalias** entre inspeções consecutivas (por ID do tubo + posição)
3. **Calcular a taxa de corrosão real** a partir das diferenças de profundidade
4. **Otimizar hiperparâmetros** com GridSearch/RandomSearch
5. **Avaliar modelos probabilísticos** para estimar incerteza nas previsões